In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score


import sklearn.linear_model as linear_model
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = LabelEncoder()

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()
train

,dri_score,psych_disturb,cyto_score,diabetes,hla_match_c_high,hla_high_res_8,tbi_status,arrhythmia,hla_low_res_6,graft_type,...,hepatic_mild,tce_div_match,donor_related,melphalan_dose,hla_low_res_8,cardiac,hla_match_drb1_high,pulm_moderate,hla_low_res_10,efs
0,7,0,7,0,0.0,0.0,0,0,6.0,0,...,0,4,2,1,8.0,0,2.0,0,10.0,0.0
1,2,0,1,0,2.0,8.0,6,0,6.0,1,...,0,3,1,1,8.0,0,2.0,2,10.0,1.0
2,7,0,7,0,2.0,8.0,0,0,6.0,0,...,0,3,1,1,8.0,0,2.0,0,10.0,0.0
3,0,0,1,0,2.0,8.0,0,0,6.0,0,...,2,3,2,1,8.0,0,2.0,0,10.0,0.0
4,0,0,7,0,2.0,8.0,0,0,6.0,1,...,0,3,1,0,8.0,0,2.0,0,10.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28795,3,3,0,0,2.0,8.0,0,0,6.0,1,...,3,0,3,1,8.0,3,2.0,0,10.0,0.0
28796,0,0,5,2,1.0,4.0,0,0,5.0,1,...,0,1,1,1,6.0,2,1.0,2,8.0,1.0
28797,9,3,5,3,2.0,8.0,0,3,6.0,1,...,3,1,2,1,8.0,3,2.0,0,10.0,0.0
28798,7,0,5,0,1.0,4.0,0,0,3.0,1,...,0,3,1,0,4.0,0,1.0,0,5.0,0.0


In [7]:
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats.distributions as dists

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size = 0.3, random_state = 0)

In [8]:
#space_lgbm={'colsample_bytree': dists.uniform(0, 1),
#            'drop_rate': dists.uniform(0, 1),
#            'learning_rate': dists.uniform(0, 1),
#            'max_bin': numpy.arange(1, 10000),
#            'max_depth': numpy.arange(1, 10000),
#            'max_drop': numpy.arange(1, 10000),
#            'min_child_samples': numpy.arange(1, 10000),
#            'min_data_in_leaf': numpy.arange(1, 10000),
#            'num_leaves': numpy.arange(1, 10000),
#            'n_estimators': numpy.arange(1000, 30000),
#            'reg_alpha': dists.uniform(0, 1),
#            'reg_lambda': dists.uniform(0, 1),
#            'skip_drop': dists.uniform(0, 1),
#            'verbosity': [-1]
#           }

#print(space_lgbm['n_estimators'])

#lgbm=LGBMRegressor()
#lgbm_cv = RandomizedSearchCV(lgbm, param_distributions=space_lgbm, cv=10, verbose=3)
#lgbm_cv.fit(X_train, y_train)

#lgbm_hyperparams=lgbm_cv.best_params_
#print("Tuned Decision Tree Parameters: {}".format(lgbm_cv.best_params_))
#print("Best score is {}".format(lgbm_cv.best_score_))

In [9]:
#space_Cat={
#            'learning_rate': dists.uniform(0, 1),
#            'max_depth': numpy.arange(1, 16),
#            'n_estimators': numpy.arange(1000, 30000),
#            'logging_level': ['Silent']
#           }

#print(space_Cat['n_estimators'])

#Cat=CatBoostRegressor()
#Cat_cv = RandomizedSearchCV(Cat, param_distributions=space_Cat, cv=10, verbose=3)
#Cat_cv.fit(X_train, y_train)

#Cat_hyperparams=Cat_cv.best_params_
#print("Tuned Decision Tree Parameters: {}".format(Cat_cv.best_params_))
#print("Best score is {}".format(Cat_cv.best_score_))

In [10]:
#space={ 'n_estimators': numpy.arange(1000, 20000),
#        'max_depth': numpy.arange(3, 30),
#        'gamma': numpy.arange(1,9),
#        'reg_alpha' : numpy.arange(40,180),
#        'reg_lambda' : dists.uniform(0, 1),
#        'colsample_bytree' : dists.uniform(0.5, 1),
#        'min_child_weight' : numpy.arange(0, 10),
#        'subsample' : dists.uniform(0, 1),
#        'learning_rate': dists.uniform(0, 1)
#    }
##print(space['n_estimators'])

#XGB=XGBRegressor()
#tree_cv = RandomizedSearchCV(XGB, param_distributions=space, cv=10, verbose=3)
#tree_cv.fit(X_train, y_train)

#XGB_hyperparams=tree_cv.best_params_
#print("Tuned Decision Tree Parameters: {}".format(tree_cv.best_params_))
#print("Best score is {}".format(tree_cv.best_score_))

In [11]:
lgbm_hyperparams={ 'colsample_bytree': 0.3394676841949491,
                   'drop_rate': 0.005229247069083676,
                   'learning_rate': 0.08939376710901481,
                   'max_bin': 3192,
                   'max_depth': 5655,
                   'max_drop': 3654,
                   'min_child_samples': 9011,
                   'min_data_in_leaf': 1320,
                   'n_estimators': 5619,
                   'num_leaves': 8002,
                   'objective': 'regression_l1',
                   'reg_alpha': 0.3092258349709437,
                   'reg_lambda': 0.5130123398875309,
                   'skip_drop': 0.5672900003846416,
                   'verbosity': -1}

XGB_hyperparams = {'colsample_bytree': 0.6679438268550072,
                   'gamma': 2,
                   'learning_rate': 0.19750405984350627,
                   'max_depth': 20,
                   'min_child_weight': 9,
                   'n_estimators': 2338,
                   'objective': 'binary:logistic',
                   'reg_alpha': 44,
                   'reg_lambda': 0.5688906203980144,
                   'subsample': 0.5716652066840949}



lightgbm = AdaBoostRegressor(LGBMRegressor(**lgbm_hyperparams),
                             random_state=42, 
                             n_estimators=4,
                             learning_rate=0.1)

xgboost = AdaBoostRegressor(XGBRegressor(n_estimators=int(XGB_hyperparams['n_estimators']),  
                                         learning_rate=XGB_hyperparams['learning_rate'],
                                         colsample_bytree=XGB_hyperparams['colsample_bytree'], 
                                         subsample=XGB_hyperparams['subsample'], 
                                         objective='binary:logistic',
                                         min_child_weight=XGB_hyperparams['min_child_weight'],
                                         gamma=XGB_hyperparams['gamma'],
                                         max_depth=int(XGB_hyperparams['max_depth']),
                                         reg_alpha=XGB_hyperparams['reg_alpha'],
                                         reg_lambda=XGB_hyperparams['reg_lambda']),
                            random_state=42, 
                            n_estimators=4,
                            learning_rate=0.1)

#reg:logistic
#binary:logitraw
#binary:logistic

In [12]:
model = StackingRegressor(estimators=[('xgboost', XGBRegressor()), 
                                      ('lightgbm', LGBMRegressor()), 
                                      ('ridgecv', RidgeCV()),
                                      ('lasso', LassoCV()), 
                                      ('elasticnet', ElasticNetCV()), 
                                      ('gbr', GradientBoostingRegressor())])


In [13]:
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007987 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 20160, number of used features: 57
[LightGBM] [Info] Start training from score 0.540575
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007459 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 811
[LightGBM] [Info] Number of data points in the train set: 16128, number of used features: 57
[LightGBM] [Info] Start training from score 0.542783
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007236 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough,

StackingRegressor(estimators=[('xgboost',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learnin...
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=None, ...)),
                              ('lightgbm', LGBMRegressor()),
                              ('ridgecv', RidgeCV()), ('lasso', LassoCV()),
                              ('elasticnet', ElasticNetCV()),
                              ('gbr', GradientBoostingRegressor())])

In [14]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']

In [15]:
test_predictions = model.predict(test)

mean_absolute_error(y_test, model.predict(X_test))

0.40363758324862303

In [16]:
0.4312733828771376

0.4312733828771376

In [17]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.302701
1,28801,0.767373
2,28802,0.235342


In [18]:
submission.to_csv('submission.csv', index = False)